In [ ]:
# import relavant modules
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

target = 'bandgap'

cwd = os.getcwd()

file_path = f"{cwd}\\pkl\\{target}_results"

# create directory to save feature statistics plots
path_to_save = f"{file_path}\\feature_statistics"
if not os.path.exists(path_to_save):
    os.makedirs(path_to_save)

# increase text size for better readability
plt.rcParams.update({'font.size': 18}) 

# load initial features .pkl file using joblib
df_init = joblib.load(f"{file_path}\\df_train_{target}_scaled.pkl")
print("Initial features loaded successfully.")

# load engineered features .pkl file using joblib
df_eng = joblib.load(f"{file_path}\\df_train_{target}_engineered.pkl")
print("Engineered features loaded successfully.")

# load most relevant features from pkl file
most_relevant_features = joblib.load(f"{file_path}\\features_selected_from_RFE_{target}.pkl")
print("Most relevant features loaded successfully.")

# drop the target column from the dataframe for intial features
df_init = df_init.drop(columns=[target,'is_experimental'])
print("Target column dropped successfully.")

# drop the target column from the dataframe for engineered features
df_eng = df_eng.drop(columns=[target,'is_experimental'])
print("Target column dropped successfully.")

# drop all columns not in most_relevant_features
df_rfe = df_eng[most_relevant_features]
print("Irrelevant features dropped successfully.")

print(df_rfe.head())


In [ ]:
# write shape and statistical analysis of all three dataframes to a text file
with open(f"{path_to_save}\\{target}_feature_statistics.txt", "w") as f:
    f.write("Initial features dataframe shape:\n")
    f.write(str(df_init.shape) + "\n\n")
    f.write("Engineered features dataframe shape:\n")
    f.write(str(df_eng.shape) + "\n\n")
    f.write("RFE selected features dataframe shape:\n")
    f.write(str(df_rfe.shape) + "\n\n")
    
    # f.write("Statistical analysis of initial features dataframe:\n")
    # f.write(str(df_init.describe()) + "\n\n")
    # f.write("Statistical analysis of engineered features dataframe:\n")
    # f.write(str(df_eng.describe()) + "\n\n")
    # f.write("Statistical analysis of RFE selected features dataframe:\n")
    # f.write(str(df_rfe.describe()) + "\n\n")

# print the shape of the features dataframe
print("Shape of the features dataframe:")
print(df_rfe.shape)

# perform statistical analysis on the features dataframe
print("Statistical analysis of the features dataframe:")
print(df_rfe.describe())


In [ ]:
# -------------------------
# 1. Compute sparsity metrics
# -------------------------
overall_sparsity_init = (df_init == 0).sum().sum() / df_init.size
feature_sparsity_init = (df_init == 0).sum(axis=0) / len(df_init)
row_sparsity_init = (df_init == 0).sum(axis=1) / df_init.shape[1]

overall_sparsity_rfe = (df_rfe == 0).sum().sum() / df_rfe.size
feature_sparsity_rfe = (df_rfe == 0).sum(axis=0) / len(df_rfe)
row_sparsity_rfe = (df_rfe == 0).sum(axis=1) / df_rfe.shape[1]

overall_sparsity_eng = (df_eng == 0).sum().sum() / df_eng.size
feature_sparsity_eng = (df_eng == 0).sum(axis=0) / len(df_eng)
row_sparsity_eng = (df_eng == 0).sum(axis=1) / df_eng.shape[1]

#write the statistical analysis to a text file without overwriting previous content
with open(f"{path_to_save}\\{target}_feature_statistics.txt", "a") as f:
    f.write(f"\n\nOverall sparsity (Initial Features): {overall_sparsity_init:.2%}")
    f.write("\nFeature sparsity summary (Initial Features):\n")
    f.write(feature_sparsity_init.describe().__str__())
    f.write("\nRow sparsity summary (Initial Features):\n")
    f.write(row_sparsity_init.describe().__str__())

    f.write(f"\n\nOverall sparsity (RFE Features): {overall_sparsity_rfe:.2%}")
    f.write("\nFeature sparsity summary (RFE Features):\n")
    f.write(feature_sparsity_rfe.describe().__str__())
    f.write("\nRow sparsity summary (RFE Features):\n")
    f.write(row_sparsity_rfe.describe().__str__())

    f.write(f"\n\nOverall sparsity (Engineered Features): {overall_sparsity_eng:.2%}")
    f.write("\nFeature sparsity summary (Engineered Features):\n")
    f.write(feature_sparsity_eng.describe().__str__())
    f.write("\nRow sparsity summary (Engineered Features):\n")
    f.write(row_sparsity_eng.describe().__str__())

In [ ]:
# -------------------------
# 2. Histogram of feature sparsity
# -------------------------

# Define common bin edges (25 bins from 0 to 1)
bins = np.linspace(0, 1, 26)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

# Initial Features
axes[0].hist(feature_sparsity_init, bins=bins, edgecolor='black', density=True, alpha=0.7)
axes[0].set_title("Initial Features")
axes[0].set_xlabel("Sparsity (fraction of zeros)")
axes[0].set_ylabel("Density")

# Engineered Features
axes[1].hist(feature_sparsity_eng, bins=bins, edgecolor='black', density=True, alpha=0.7)
axes[1].set_title("Engineered Features")
axes[1].set_xlabel("Sparsity (fraction of zeros)")

# RFE Selected Features
axes[2].hist(feature_sparsity_rfe, bins=bins, edgecolor='black', density=True, alpha=0.7)
axes[2].set_title("RFE Selected Features")
axes[2].set_xlabel("Sparsity (fraction of zeros)")

plt.tight_layout()
plt.savefig(f"{path_to_save}\\{target}_distribution_of_sparsest_features.png", bbox_inches='tight')
plt.show()



In [ ]:
# -------------------------
# 3. Ranked bar plot of top-k sparsest features
# -------------------------
top_k = 20
sorted_features = feature_sparsity_rfe.sort_values(ascending=False).head(top_k)

plt.figure(figsize=(10,6))
sorted_features.plot(kind="barh", color="skyblue", edgecolor="black")
plt.gca().invert_yaxis()  # highest sparsity at top
# plt.title(f"Top {top_k} Sparsest Features")
plt.xlabel("Sparsity (fraction of zeros)")
plt.ylabel("Feature")

# save to file
plt.savefig(f"{path_to_save}\\{target}_top_{top_k}_sparsest_features_rfe.png", bbox_inches='tight')
plt.show()


# # Plot feature relevance scores
#         sns.set(rc={'figure.figsize': (10, 10)})
#         sns.set(font_scale=2.0)
#         sns.set_style("ticks")
#         fig = sns.barplot(x='relevance_score', y='feature', data=feature_score[:no_of_features_to_plot],color='skyblue')

In [ ]:
# -------------------------
# 4a. Heatmap (RFE binarized version, zero vs nonzero)
# -------------------------
subset_rfe = df_rfe.sample(n=500, random_state=42)  # sample 500 rows
binary_matrix_rfe = (subset_rfe == 0).astype(int)

# cap at 1 for better visualization
subset_rfe[subset_rfe > 1] = 1
binary_matrix_rfe[binary_matrix_rfe > 1] = 1

plt.figure(figsize=(12,6))
plt.imshow(binary_matrix_rfe.T, aspect='auto', cmap="Greys", interpolation="nearest")
# plt.title("Heatmap of Zero Entries (500 Sampled Rows)")
plt.xlabel("Samples")
plt.ylabel("Features")
plt.colorbar(label="Zero=1, Nonzero=0")
plt.savefig(f"{path_to_save}\\{target}_rfe_heatmap_binary.png", bbox_inches='tight')
plt.show()

# -------------------------
# 4b. Heatmap (RFE non-binarized version with real values)
# -------------------------
plt.figure(figsize=(12,6))
plt.imshow(subset_rfe.T, aspect='auto', cmap="viridis", interpolation="nearest")
# plt.title("Heatmap of Feature Values (500 Sampled Rows)")
plt.xlabel("Samples")
plt.ylabel("Features")
plt.colorbar(label="Feature Value")
plt.savefig(f"{path_to_save}\\{target}_rfe_heatmap_values.png", bbox_inches='tight')
plt.show()

# -------------------------
# 4c. Heatmap (ENG binarized version, zero vs nonzero)
# -------------------------
subset_eng = df_eng.sample(n=500, random_state=42)  # sample 500 rows
binary_matrix_eng = (subset_eng == 0).astype(int)

# cap at 1 for better visualization
subset_eng[subset_eng > 1] = 1
binary_matrix_eng[binary_matrix_eng > 1] = 1

plt.figure(figsize=(12,6))
plt.imshow(binary_matrix_eng.T, aspect='auto', cmap="Greys", interpolation="nearest")
# plt.title("Heatmap of Zero Entries (500 Sampled Rows)")
plt.xlabel("Samples")
plt.ylabel("Features")
plt.colorbar(label="Zero=1, Nonzero=0")
plt.savefig(f"{path_to_save}\\{target}_eng_heatmap_binary.png", bbox_inches='tight')
plt.show()

# -------------------------
# 4d. Heatmap (ENG non-binarized version with real values)
# -------------------------
plt.figure(figsize=(12,6))
plt.imshow(subset_eng.T, aspect='auto', cmap="viridis", interpolation="nearest")
# plt.title("Heatmap of Feature Values (500 Sampled Rows)")
plt.xlabel("Samples")
plt.ylabel("Features")
plt.colorbar(label="Feature Value")
plt.savefig(f"{path_to_save}\\{target}_eng_heatmap_values.png", bbox_inches='tight')
plt.show()

# -------------------------
# 4e. Heatmap (INIT binarized version, zero vs nonzero)
# -------------------------
subset_init = df_init.sample(n=500, random_state=42)  # sample 500 rows
binary_matrix_init = (subset_init == 0).astype(int)

# cap at 1 for better visualization
subset_init[subset_init > 1] = 1
binary_matrix_init[binary_matrix_init > 1] = 1

plt.figure(figsize=(12,6))
plt.imshow(binary_matrix_init.T, aspect='auto', cmap="Greys", interpolation="nearest")
# plt.title("Heatmap of Zero Entries (500 Sampled Rows)")
plt.xlabel("Samples")
plt.ylabel("Features")
plt.colorbar(label="Zero=1, Nonzero=0")
plt.savefig(f"{path_to_save}\\{target}_init_heatmap_binary.png", bbox_inches='tight')
plt.show()

# -------------------------
# 4f. Heatmap (INIT non-binarized version with real values)
# -------------------------
plt.figure(figsize=(12,6))
plt.imshow(subset_init.T, aspect='auto', cmap="viridis", interpolation="nearest")
# plt.title("Heatmap of Feature Values (500 Sampled Rows)")
plt.xlabel("Samples")
plt.ylabel("Features")
plt.colorbar(label="Feature Value")
plt.savefig(f"{path_to_save}\\{target}_init_heatmap_values.png", bbox_inches='tight')
plt.show()


In [ ]:
# -------------------------
# 5. Cumulative coverage curve
# -------------------------
nonzero_counts_rfe = (df_rfe != 0).sum(axis=0)
sorted_counts_rfe = nonzero_counts_rfe.sort_values(ascending=False)
cumulative_rfe = sorted_counts_rfe.cumsum() / sorted_counts_rfe.sum()

nonzero_counts_eng = (df_eng != 0).sum(axis=0)
sorted_counts_eng = nonzero_counts_eng.sort_values(ascending=False)
cumulative_eng = sorted_counts_eng.cumsum() / sorted_counts_eng.sum()

nonzero_counts_init = (df_init != 0).sum(axis=0)
sorted_counts_init = nonzero_counts_init.sort_values(ascending=False)
cumulative_init = sorted_counts_init.cumsum() / sorted_counts_init.sum()

plt.figure(figsize=(8,5))
plt.plot(range(1, len(sorted_counts_rfe)+1), cumulative_rfe, marker=".", label="RFE Features")
plt.plot(range(1, len(sorted_counts_eng)+1), cumulative_eng, marker=".", label="Engineered Features")
plt.plot(range(1, len(sorted_counts_init)+1), cumulative_init, marker=".", label="Initial Features")
plt.legend()
# plt.title("Cumulative Coverage by Features")
plt.xlabel("Number of features included")
plt.ylabel("Fraction of nonzero entries")
plt.grid(True, alpha=0.3)
plt.savefig(f"{path_to_save}\\{target}_cumulative_coverage.png", bbox_inches='tight')
plt.show()